# 09_01 multi-doc 기준선

G3 동결 세트(600건, D-15)에서 베이스라인 dense 검색(KURE-v1, 512/64 전체 인덱스)의 Recall·Complete·Hit@k를 보고, SPEC 지표의 k(5, 10)를 바꿀지 판단할 근거를 모은다. 판정에 쓰지 않는 보조 관찰이다(SPEC "보조 관찰"). 명세는 `docs/slices/09-multidoc-eval.md`.

In [1]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from rag.eval.bootstrap import paired_bootstrap
from rag.eval_multi.metrics import (
    load_gold, metric_table, per_query, ranked_docs, read_run, sha256,
)

In [2]:
ROOT = Path.cwd().parent
CFG = OmegaConf.load(ROOT / "configs/config.yaml")
SPEC = OmegaConf.load(ROOT / "configs/eval/spec_v1.yaml")
FROZEN = ROOT / CFG.paths.multidoc_frozen
RUN = ROOT / CFG.paths.runs_dir / "kure-v1-fixed-512-64-multidoc.csv"
JUDGE_V1 = ROOT / CFG.paths.multidoc_judge_out / "scope_v1"
KS = list(CFG.multidoc.eval.ks)
BS = SPEC.bootstrap
KS, dict(BS)

([5, 10],
 {'n_resamples': 10000,
  'unit': 'conference_number',
  'seed': 20260929,
  'alpha': 0.05,
  'min_gain': 0.01})

In [3]:
def level_ci(values, groups):
    """지표 수준의 95% 구간. 질의 값을 그대로 paired_bootstrap에 넣는다(기준 0과의 차이)."""
    _, lo, hi = paired_bootstrap(np.asarray(values), np.asarray(groups),
                                 BS.n_resamples, BS.seed, BS.alpha)
    return lo, hi

## 데이터

In [4]:
gold = load_gold(FROZEN)
gold["n_gold"] = gold["gold"].map(len)
print(len(gold), sha256(FROZEN))
gold.head(3)

600 0c385a2bad74c6244e203fb6a428993badf4afb04db4848276a3b48d1a874436


,type,pool_id,gold,n_gold
qid,,,,
md-conf-0002,conf,conf-0002,"frozenset({047419-0c4826dfc8, 047419-0c68adf282})",2
md-conf-0003,conf,conf-0003,"frozenset({050718-437a89bff5, 050718-db2bf0856...",3
md-conf-0004,conf,conf-0004,"frozenset({047413-48e2026591, 047413-5965de10bd})",2


In [5]:
pd.crosstab(gold["type"], gold["n_gold"], margins=True)

n_gold,2,3,4,5,All
type,,,,,
conf,166,54,11,2,233
law,93,43,11,5,152
questioner,132,46,28,9,215
All,391,143,50,16,600


## 검색

`rag.search index.scope=full search.queries=multidoc`의 순위 파일이다. 질의 임베딩은 Colab T4(세션 meet-rag-s09q)에서 만들었고, 상위 200청크 안에서 문서 10개를 모두 채워 전체 정렬로 넘어간 질의는 0건이었다.

In [6]:
run = read_run(RUN)
docs = ranked_docs(run, gold, max(KS))
print(sha256(RUN), run.shape, docs.map(len).value_counts().to_dict())

8f14dc808c16b94afac5b4167c7e63088a27410290e1f113135d537643bf8289 (6000, 3) {10: 600}


In [7]:
# 질의 하나의 상위 5개와 정답
qid = gold.index[0]
pd.DataFrame({"doc_id": docs[qid][:5], "정답": [d in gold.loc[qid, "gold"] for d in docs[qid][:5]]})

,doc_id,정답
0,047419-0c68adf282,True
1,047419-0c4826dfc8,True
2,046890-25733ff95c,False
3,049982-61b185ae39,False
4,052625-0b0a273a8d,False


## 평가

Recall@k는 정답 중 상위 k에 든 비율의 평균, Complete@k는 정답이 모두 든 질의 비율, Hit@k는 정답이 하나라도 든 질의 비율이다.

In [8]:
scores = per_query(docs, gold, KS)
table = metric_table(scores, gold, ["type"])
table.round(4)

,group,value,n,recall@5,complete@5,hit@5,recall@10,complete@10,hit@10
0,all,all,600,0.4657,0.3150,0.6183,0.5232,0.3900,0.6600
1,type,conf,233,0.8288,0.6567,0.9828,0.8847,0.7682,0.9828
2,type,law,152,0.2990,0.1053,0.5132,0.3720,0.1579,0.6118
3,type,questioner,215,0.1900,0.0930,0.2977,0.2384,0.1442,0.3442


In [9]:
# 지표 수준의 95% 구간(재표본 단위 pool_id = 질의)
rows = []
for name, idx in [("all", gold.index), *gold.groupby("type").groups.items()]:
    for m in scores.columns:
        lo, hi = level_ci(scores.loc[idx, m], gold.loc[idx, "pool_id"])
        rows.append({"group": name, "metric": m, "mean": scores.loc[idx, m].mean(),
                     "lo": lo, "hi": hi})
ci = pd.DataFrame(rows)
ci.pivot(index="metric", columns="group", values="mean").round(3).astype(str) + " [" + \
    ci.pivot(index="metric", columns="group", values="lo").round(3).astype(str) + ", " + \
    ci.pivot(index="metric", columns="group", values="hi").round(3).astype(str) + "]"

group,all,conf,law,questioner
metric,,,,
complete@10,"0.39 [0.352, 0.428]","0.768 [0.712, 0.82]","0.158 [0.105, 0.217]","0.144 [0.098, 0.191]"
complete@5,"0.315 [0.278, 0.352]","0.657 [0.597, 0.717]","0.105 [0.059, 0.158]","0.093 [0.056, 0.135]"
hit@10,"0.66 [0.622, 0.697]","0.983 [0.966, 0.996]","0.612 [0.533, 0.691]","0.344 [0.279, 0.405]"
hit@5,"0.618 [0.58, 0.657]","0.983 [0.966, 0.996]","0.513 [0.434, 0.592]","0.298 [0.237, 0.358]"
recall@10,"0.523 [0.489, 0.558]","0.885 [0.855, 0.913]","0.372 [0.315, 0.429]","0.238 [0.191, 0.288]"
recall@5,"0.466 [0.432, 0.499]","0.829 [0.795, 0.861]","0.299 [0.246, 0.354]","0.19 [0.148, 0.234]"


In [10]:
# 정답 문서 수별
metric_table(scores, gold, ["n_gold"]).query("group != 'all'").round(4)

,group,value,n,recall@5,complete@5,hit@5,recall@10,complete@10,hit@10
1,n_gold,2,391,0.5320,0.4092,0.6547,0.5870,0.4834,0.6905
2,n_gold,3,143,0.3963,0.1818,0.6154,0.4685,0.2797,0.6713
3,n_gold,4,50,0.2350,0.0600,0.3800,0.2650,0.0800,0.4200
4,n_gold,5,16,0.1875,0.0000,0.5000,0.2625,0.0625,0.5625


## 정답 문서의 순위

In [11]:
ranks = []
for q, row in gold.iterrows():
    pos = {d: i for i, d in enumerate(docs[q][:10], 1)}
    ranks += [{"qid": q, "type": row["type"], "rank": pos.get(d)} for d in row["gold"]]
ranks = pd.DataFrame(ranks)
ranks["구간"] = pd.cut(ranks["rank"].fillna(11), [0, 1, 5, 10, 11],
                       labels=["1", "2-5", "6-10", "10위 밖"])
pd.crosstab(ranks["type"], ranks["구간"], normalize="index", margins=True).round(3)

구간,1,2-5,6-10,10위 밖
type,,,,
conf,0.376,0.432,0.064,0.128
law,0.125,0.182,0.073,0.620
questioner,0.066,0.089,0.041,0.803
All,0.195,0.239,0.058,0.508


In [12]:
# k=1..10의 Recall·Complete(k를 바꿀 때 어떻게 움직이는지)
curve = per_query(docs, gold, range(1, 11))
rec = curve.filter(like="recall@").groupby(gold["type"]).mean().T
com = curve.filter(like="complete@").groupby(gold["type"]).mean().T
rec["all"] = curve.filter(like="recall@").mean()
com["all"] = curve.filter(like="complete@").mean()
pd.concat({"recall": rec, "complete": com}, axis=1).round(3)

recall                          complete                         
type          conf    law questioner    all     conf    law questioner    all
recall@1     0.397  0.129      0.082  0.216      NaN    NaN        NaN    NaN
recall@2     0.660  0.213      0.132  0.357      NaN    NaN        NaN    NaN
recall@3     0.752  0.251      0.157  0.412      NaN    NaN        NaN    NaN
recall@4     0.794  0.276      0.174  0.441      NaN    NaN        NaN    NaN
recall@5     0.829  0.299      0.190  0.466      NaN    NaN        NaN    NaN
recall@6     0.843  0.311      0.194  0.476      NaN    NaN        NaN    NaN
recall@7     0.866  0.324      0.202  0.491      NaN    NaN        NaN    NaN
recall@8     0.870  0.344      0.226  0.506      NaN    NaN        NaN    NaN
recall@9     0.875  0.354      0.230  0.512      NaN    NaN        NaN    NaN
recall@10    0.885  0.372      0.238  0.523      NaN    NaN        NaN    NaN
complete@1     NaN    NaN        NaN    NaN    0.000  0.000      0.000  0.000
complete@2     NaN    NaN        NaN    NaN    0.373  0.046      0.047  0.173
complete@3     NaN    NaN        NaN    NaN    0.511  0.079      0.070  0.243
complete@4     NaN    NaN        NaN    NaN    0.584  0.086      0.079  0.277
complete@5     NaN    NaN        NaN    NaN    0.657  0.105      0.093  0.315
complete@6     NaN    NaN        NaN    NaN    0.678  0.112      0.093  0.325
complete@7     NaN    NaN        NaN    NaN    0.721  0.112      0.098  0.343
complete@8     NaN    NaN        NaN    NaN    0.734  0.145      0.126  0.367
complete@9     NaN    NaN        NaN    NaN    0.742  0.151      0.130  0.373
complete@10    NaN    NaN        NaN    NaN    0.768  0.158      0.144  0.390

## 상위 문서가 후보 집합 안에 있는가

후보 집합(`pool_doc_ids`)은 질의를 만든 묶음(같은 회의·법안·질의자)의 문서다. 상위 10개 중 후보 집합 문서의 비율이 높은데 정답을 놓치면 같은 묶음 안의 다른 문서를 고른 것이고, 낮으면 묶음 밖 문서로 간 것이다. 묶음 밖 문서가 실제로 관련 있어도 정답으로 세지 않는다(SPEC 해석의 한계).

In [13]:
pool = {}
with FROZEN.open(encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        pool[r["qid"]] = set(r["pool_doc_ids"])
in_pool = pd.Series({q: np.mean([d in pool[q] for d in docs[q][:10]]) for q in gold.index})
pool_size = pd.Series({q: len(pool[q]) for q in gold.index})
pd.DataFrame({"상위10 중 후보 집합 비율": in_pool.groupby(gold["type"]).mean(),
              "후보 집합 크기 중앙값": pool_size.groupby(gold["type"]).median()}).round(3)

,상위10 중 후보 집합 비율,후보 집합 크기 중앙값
type,,
conf,0.246,14.0
law,0.126,3.0
questioner,0.087,6.0


## law 단서

동결된 `law` 152건 중 `scope_v1`·`scope_v2`로 두 번 판정한 질의와 `scope_v2`로 한 번만 판정한 질의를 나눈다(D-15). 한 번만 판정한 쪽에 다시 판정하면 걸릴 질의가 남을 수 있다.

In [14]:
twice = {f.stem for f in JUDGE_V1.glob("*.json")}
law = gold["type"] == "law"
judged = pd.Series(np.where(gold.index.isin(twice), "두 번", "한 번"), index=gold.index)[law]
scores[law].groupby(judged).mean().assign(n=judged.value_counts()).round(4)

,recall@5,complete@5,hit@5,recall@10,complete@10,hit@10,n
두 번,0.2599,0.0988,0.4321,0.3218,0.1358,0.5309,81
한 번,0.3437,0.1127,0.6056,0.4293,0.1831,0.7042,71


## 실패 사례

Hit@10이 0인 질의(정답이 상위 10위에 하나도 없음) 몇 건. 질의 원문은 소량만 출력한다.

In [15]:
miss = scores.index[scores["hit@10"] == 0]
print(len(miss), Counter(gold.loc[miss, "type"]))
text = {}
with FROZEN.open(encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        text[r["qid"]] = r["query"]
pick = [gold.loc[miss][gold.loc[miss, "type"] == t].index[0] for t in ("law", "questioner")]
for q in pick:
    print(q, text[q])
    print("  정답:", sorted(gold.loc[q, "gold"]))
    print("  1위:", docs[q][0], "후보 집합 안" if docs[q][0] in pool[q] else "후보 집합 밖")

204 Counter({'questioner': 141, 'law': 59, 'conf': 4})
md-law-0000 방송법 일부개정법률안 심사에서 2007년 9월 11일 방송통신특별위원회 제11차 회의와 2011년 12월 22일 문화체육관광방송통신위원회 법안심사소위원회 제2차 회의에서 각각 위원들이 문제로 삼은 쟁점은 무엇인가?
  정답: ['036203-9a8cf7e705', '041535-6c0bd98b86']
  1위: 051404-0effaabb4d 후보 집합 밖
md-questioner-0000 김성찬 위원이 2019년 7월 12일 농림축산식품해양수산위원회 예산결산심사소위원회(제369회 제1차)의 2019년도 제1회 추가경정예산안 심사에서 제기한 사항들은 무엇인가?
  정답: ['049232-2538ce29ed', '049232-628e88796b', '049232-cff1c4579f']
  1위: 049574-281a967dcc 후보 집합 밖


## 질의가 기대는 메타데이터

위 두 사례는 날짜·위원회·위원 이름으로 회의를 지목하는데, 인덱스는 문서 `context`(발언 본문)만 임베딩했다(S4). 아래는 정답 문서 본문에 질의 첫 낱말(법안명·위원 이름)과 연도가 있는지(`(첫 낱말, 연도)`)와, 질의에 날짜(`NNNN년`)가 있는 비율이다.

In [16]:
ctx = {}
want = set().union(*[gold.loc[q, "gold"] for q in pick])
with (ROOT / CFG.paths.corpus).open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["doc_id"] in want:
            ctx[d["doc_id"]] = d["context"]
for q in pick:
    head = text[q].split()[0]
    year = text[q][text[q].index("년") - 4 : text[q].index("년") + 1]
    print(q, {d: (head in ctx[d], year in ctx[d]) for d in sorted(gold.loc[q, "gold"])})

md-law-0000 {'036203-9a8cf7e705': (True, False), '041535-6c0bd98b86': (False, False)}
md-questioner-0000 {'049232-2538ce29ed': (True, False), '049232-628e88796b': (False, False), '049232-cff1c4579f': (False, False)}


In [18]:
has_date = pd.Series([text[q] for q in gold.index], index=gold.index).str.contains(r"\d{4}년")
has_date.groupby(gold["type"]).mean()

type
conf          1.0
law           1.0
questioner    1.0
dtype: float64

## 요약

- 전체 Recall@5 0.466, Complete@5 0.315, Hit@5 0.618(@10: 0.523, 0.390, 0.660). 유형 차이가 크다: `conf` Hit@5 0.983·Complete@5 0.657, `law` 0.513·0.105, `questioner` 0.298·0.093
- 정답 문서 중 상위 10위 밖 비율은 `conf` 12.8%, `law` 62.0%, `questioner` 80.3%다. `law`·`questioner`는 정답 문서를 아예 찾지 못하는 것이 주된 실패이고, 상위 10개 중 후보 집합 문서 비율도 낮다(0.126, 0.087)
- 실패 사례의 질의는 날짜·위원회·위원 이름으로 회의를 지목하지만, 정답 문서 본문에는 연도가 없고 위원 이름도 일부 문서에만 있다. 모든 질의에 날짜가 있어(`conf` 포함) 날짜 유무로 나눠 비교할 수는 없다. `conf`는 한 회의의 쟁점을 묻고 쟁점 낱말이 본문에 있어 영향이 작아 보인다(추론). 메타데이터를 검색에 넣는 실험(H12 화자 메타데이터 등)의 관찰 대상이다
- `law`는 한 번만 판정한 71건이 두 번 판정한 81건보다 점수가 높다(Hit@10 0.704 대 0.531). 재판정에서 더 걸러졌을 질의가 남아 있다는 단서와 방향이 맞는지는 이 표로 판정할 수 없다
- k: 5에서 10으로 늘려도 Recall은 `law` 0.299→0.372, `questioner` 0.190→0.238로 조금 오른다. 정답 문서가 최대 5개라 Complete@5까지는 정의상 가능하고, 실패는 순위 깊이보다 문서를 찾지 못하는 데서 온다
